# 🔍 CNN 기반 불량 분류 (Defect Classification)

> **학습 목표**: 딥러닝 CNN으로 제조 불량 유형을 자동 분류합니다.

---

## 📋 학습 내용

1. ✅ 불량 이미지 데이터셋 준비
2. ✅ MobileNetV2 전이학습 (Transfer Learning)
3. ✅ 모델 학습 및 검증
4. ✅ 혼동 행렬 (Confusion Matrix) 분석
5. ✅ 분류 리포트 (Precision, Recall, F1-Score)
6. ✅ 예측 결과 시각화

**소요 시간**: 약 45분  
**난이도**: ⭐⭐⭐ (고급)  
**사전 지식**: CNN 기초, 전이학습 개념

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 딥러닝 프레임워크 (TensorFlow/Keras 또는 PyTorch)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    print(f"✅ TensorFlow 버전: {tf.__version__}")
    FRAMEWORK = 'tensorflow'
except ImportError:
    print("⚠️ TensorFlow 미설치 - 설치 필요: pip install tensorflow")
    FRAMEWORK = None

# 데이터 분석
import pandas as pd
import numpy as np

# 이미지 처리
from PIL import Image
import cv2

# 평가 지표
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

# GPU 설정 (사용 가능한 경우)
if FRAMEWORK == 'tensorflow':
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ GPU 사용 가능: {len(gpus)}개")
    else:
        print("ℹ️ CPU 모드로 실행")

print("\n✅ 라이브러리 로드 완료!")

## 📂 Step 2: 불량 데이터셋 준비

In [ ]:
# 데이터 디렉토리 — KAMP 실제 데이터 우선, 없으면 샘플 폴더 폴백
# KAMP 시지트로닉스 이미지: 6개 서브디렉토리(AREA, DISTRIBUTION, FAIL, NEEDLE, PASS, SCATCH)가 클래스 역할
kamp_dir = Path('../../dataset/part1-2/시지트로닉스_이미지')
fallback_dir = Path('../data/defect_images')

if kamp_dir.exists() and any(d.is_dir() for d in kamp_dir.iterdir()):
    data_dir = kamp_dir
    print(f"✅ KAMP 실제 데이터 사용: {data_dir.resolve()}")
else:
    data_dir = fallback_dir
    print(f"ℹ️ KAMP 데이터 없음 → 샘플 폴더 사용: {data_dir}")

# 데이터셋 구조 확인
if not data_dir.exists():
    print(f"⚠️ 데이터 디렉토리 없음: {data_dir}")
    print("   샘플 데이터셋 생성 중...")
    
    data_dir.mkdir(exist_ok=True, parents=True)
    
    # 불량 유형별 폴더 생성
    classes = ['normal', 'scratch', 'contamination', 'damage']
    for cls in classes:
        cls_dir = data_dir / cls
        cls_dir.mkdir(exist_ok=True)
        
        # 샘플 이미지 생성 (각 클래스당 10개)
        for i in range(10):
            img_array = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(img_array)
            img.save(cls_dir / f'{cls}_{i:02d}.png')
    
    print(f"✅ 샘플 데이터셋 생성 완료")
else:
    classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
    if len(classes) == 0:
        # 클래스 폴더가 없으면 기본 클래스 생성
        classes = ['normal', 'scratch', 'contamination', 'damage']
        for cls in classes:
            cls_dir = data_dir / cls
            cls_dir.mkdir(exist_ok=True)
            for i in range(10):
                img_array = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
                img = Image.fromarray(img_array)
                img.save(cls_dir / f'{cls}_{i:02d}.png')

print(f"📊 데이터셋 구조:")
print(f"   루트: {data_dir}")
print(f"   클래스: {len(classes)}개")

# 클래스별 샘플 개수 확인
class_counts = {}
for cls in classes:
    cls_dir = data_dir / cls
    if cls_dir.exists():
        images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        class_counts[cls] = len(images)
        print(f"   - {cls}: {len(images)}개")

total_images = sum(class_counts.values())
print(f"\n✅ 총 이미지: {total_images}개")

In [ ]:
# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, cls in enumerate(classes):
    cls_dir = data_dir / cls
    images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    
    if len(images) > 0:
        sample_img = Image.open(images[0])
        axes[idx].imshow(sample_img)
        axes[idx].set_title(f'{cls} ({class_counts[cls]}개)', fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, 'No Image', ha='center', va='center')
        axes[idx].set_title(cls, fontsize=12, fontweight='bold')
        axes[idx].axis('off')

# 빈 서브플롯 숨기기
for idx in range(len(classes), 8):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 🤖 Step 3: MobileNetV2 전이학습 모델 구축

In [ ]:
if FRAMEWORK == 'tensorflow':
    # 하이퍼파라미터
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 16
    EPOCHS = 10
    NUM_CLASSES = len(classes)
    
    print(f"⚙️ 하이퍼파라미터:")
    print(f"   이미지 크기: {IMG_SIZE}")
    print(f"   배치 크기: {BATCH_SIZE}")
    print(f"   에폭: {EPOCHS}")
    print(f"   클래스 수: {NUM_CLASSES}")
    
    # 데이터 증강 (Data Augmentation)
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        validation_split=0.2  # 20% 검증 데이터
    )
    
    # 학습 데이터 생성
    train_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )
    
    # 검증 데이터 생성
    val_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
    
    print(f"\n✅ 데이터 생성기 준비 완료")
    print(f"   학습 샘플: {train_generator.samples}개")
    print(f"   검증 샘플: {val_generator.samples}개")
    print(f"   클래스 매핑: {train_generator.class_indices}")
else:
    print("⚠️ TensorFlow가 설치되지 않아 모델 구축을 건너뜁니다.")

In [ ]:
if FRAMEWORK == 'tensorflow':
    # MobileNetV2 기본 모델 (ImageNet 가중치)
    print("🔄 MobileNetV2 모델 로드 중...")
    
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,  # 분류 헤드 제거
        weights='imagenet'
    )
    
    # 기본 모델 가중치 동결 (Fine-tuning 전략)
    base_model.trainable = False
    
    # 분류 헤드 추가
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    
    # 모델 컴파일
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"\n✅ 모델 구축 완료!")
    print(f"   총 파라미터: {model.count_params():,}개")
    print(f"   학습 가능 파라미터: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}개")
    
    # 모델 구조 요약
    model.summary()
else:
    print("⚠️ TensorFlow가 설치되지 않아 모델 생성을 건너뜁니다.")

## 🎓 Step 4: 모델 학습

In [ ]:
if FRAMEWORK == 'tensorflow' and train_generator.samples > 0:
    # 콜백 설정
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            '../outputs/best_defect_model.h5',
            monitor='val_accuracy',
            save_best_only=True
        )
    ]
    
    print("🔄 모델 학습 시작...")
    print(f"   에폭: {EPOCHS}")
    print(f"   Early Stopping: patience=3")
    
    # 학습
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✅ 모델 학습 완료!")
else:
    print("⚠️ 학습 데이터가 부족하거나 TensorFlow가 없어 학습을 건너뜁니다.")
    history = None

In [ ]:
if history is not None:
    # 학습 곡선 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_title('모델 정확도 (Accuracy)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_title('모델 손실 (Loss)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 최종 성능
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    
    print(f"\n📊 최종 성능:")
    print(f"   Train Accuracy: {final_train_acc:.4f}")
    print(f"   Validation Accuracy: {final_val_acc:.4f}")
    
    if final_train_acc - final_val_acc > 0.1:
        print(f"\n⚠️ 과적합 가능성 (Train-Val 차이: {final_train_acc - final_val_acc:.4f})")
    else:
        print(f"\n✅ 적절한 일반화 성능")
else:
    print("⚠️ 학습 기록이 없어 시각화를 건너뜁니다.")

## 📊 Step 5: 모델 평가 (혼동 행렬)

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 검증 데이터에 대한 예측
    print("🔄 모델 예측 중...")
    
    val_generator.reset()
    y_pred_probs = model.predict(val_generator)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = val_generator.classes
    
    # 정확도
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\n📊 검증 세트 정확도: {accuracy:.4f}")
    
    # 혼동 행렬
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': '샘플 수'})
    plt.title('혼동 행렬 (Confusion Matrix)', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('예측 클래스')
    plt.ylabel('실제 클래스')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 혼동 행렬 해석:")
    print("   - 대각선: 올바르게 분류된 샘플")
    print("   - 비대각선: 잘못 분류된 샘플")
    print("   - 행: 실제 클래스")
    print("   - 열: 예측 클래스")
else:
    print("⚠️ 모델이 없어 평가를 건너뜁니다.")

## 📋 Step 6: 분류 리포트 (Precision, Recall, F1-Score)

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 분류 리포트
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    
    # DataFrame 변환
    report_df = pd.DataFrame(report).transpose()
    
    print("📋 분류 리포트:")
    print(report_df.to_string())
    
    # 클래스별 성능 시각화
    metrics = ['precision', 'recall', 'f1-score']
    class_metrics = report_df.loc[classes, metrics]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    class_metrics.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('클래스별 성능 지표', fontsize=14, fontweight='bold')
    ax.set_xlabel('클래스')
    ax.set_ylabel('점수')
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.legend(title='지표', loc='lower right')
    ax.grid(alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 성능 지표 의미:")
    print("   - Precision (정밀도): 예측이 얼마나 정확한가?")
    print("   - Recall (재현율): 실제를 얼마나 잘 찾았는가?")
    print("   - F1-Score: Precision과 Recall의 조화평균")
    print("   - Support: 각 클래스의 실제 샘플 수")
else:
    print("⚠️ 모델이 없어 리포트를 건너뜁니다.")

## 🖼️ Step 7: 예측 결과 시각화

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 샘플 예측 시각화
    num_samples = min(8, len(y_pred))
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    val_generator.reset()
    sample_batch = next(val_generator)
    
    for i in range(num_samples):
        img = sample_batch[0][i]
        true_label = classes[y_true[i]]
        pred_label = classes[y_pred[i]]
        confidence = y_pred_probs[i][y_pred[i]] * 100
        
        axes[i].imshow(img)
        
        # 제목 색상 (정답/오답)
        color = 'green' if y_true[i] == y_pred[i] else 'red'
        title = f'실제: {true_label}\n예측: {pred_label} ({confidence:.1f}%)'
        axes[i].set_title(title, fontsize=10, fontweight='bold', color=color)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 정답/오답 통계
    correct = (y_true == y_pred).sum()
    incorrect = len(y_true) - correct
    
    print(f"\n📊 예측 통계:")
    print(f"   정답: {correct}개 ({correct/len(y_true)*100:.1f}%)")
    print(f"   오답: {incorrect}개 ({incorrect/len(y_true)*100:.1f}%)")
else:
    print("⚠️ 모델이 없어 예측 시각화를 건너뜁니다.")

## 💾 Step 8: 모델 및 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

if FRAMEWORK == 'tensorflow' and history is not None:
    # 1. 모델 저장
    model_file = output_dir / '03_defect_classification_model.h5'
    model.save(model_file)
    print(f"✅ 모델 저장: {model_file}")
    
    # 2. 학습 기록 저장
    history_df = pd.DataFrame(history.history)
    history_file = output_dir / '03_training_history.csv'
    history_df.to_csv(history_file, index=False, encoding='utf-8-sig')
    print(f"✅ 학습 기록 저장: {history_file}")
    
    # 3. 분류 리포트 저장
    report_file = output_dir / '03_classification_report.csv'
    report_df.to_csv(report_file, encoding='utf-8-sig')
    print(f"✅ 분류 리포트 저장: {report_file}")
    
    # 4. 혼동 행렬 저장
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_file = output_dir / '03_confusion_matrix.csv'
    cm_df.to_csv(cm_file, encoding='utf-8-sig')
    print(f"✅ 혼동 행렬 저장: {cm_file}")
    
    print("\n🎉 불량 분류 모델 학습 및 평가 완료!")
else:
    print("⚠️ TensorFlow가 없거나 모델이 학습되지 않아 저장을 건너뜁니다.")
    print("\n💡 TensorFlow 설치: pip install tensorflow")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. 불량 이미지 데이터셋 준비 (4개 클래스)
2. MobileNetV2 전이학습 모델 구축
3. 데이터 증강 및 모델 학습
4. 혼동 행렬로 분류 성능 분석
5. Precision, Recall, F1-Score 계산
6. 예측 결과 시각화
7. 학습 모델 및 결과 저장

### 💡 핵심 인사이트
- **전이학습의 장점**:
  - ImageNet으로 사전학습된 가중치 활용
  - 적은 데이터로도 높은 정확도
  - 학습 시간 대폭 단축

- **MobileNetV2 선택 이유**:
  - 경량화: 엣지 디바이스 배포 용이
  - 빠른 추론: 실시간 불량 검사 가능
  - 우수한 성능: 정확도와 속도의 균형

- **데이터 증강 효과**:
  - 회전, 이동, 반전으로 데이터 다양화
  - 과적합 방지
  - 모델 일반화 성능 향상

- **성능 지표 해석**:
  - **Precision**: 불량 판정의 신뢰도
  - **Recall**: 실제 불량 탐지율
  - **F1-Score**: 전체적인 균형 지표

### 📚 다음 단계
- **Part 2-2**: Vision Transformer (ViT) 활용
- **Part 2-2**: YOLOv8 객체 탐지

### 🔗 참고 자료
- [MobileNetV2 논문](https://arxiv.org/abs/1801.04381)
- [전이학습 가이드](https://www.tensorflow.org/tutorials/images/transfer_learning)
- [CNN 시각화](https://poloclub.github.io/cnn-explainer/)

---

*제조AI 교육 v12 Enhanced | Part 1-2 | 2025.02*